# Parse A TPS Sequence To JSON

This notebook loads a TNMR `.tps` sequence with the repo-local parser and writes a JSON representation for inspection or debugging.

## 1. Setup

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Find the Hardware Capture repo root from a notebook or script working directory."""

    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        redstone_dir = candidate / "hardware_capture" / "rf" / "redstone"
        if (redstone_dir / "sequence.py").exists() and (redstone_dir / "waveform.py").exists():
            return candidate
    raise FileNotFoundError("Could not find the Hardware Capture repo root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REDSTONE_DIR = REPO_ROOT / "hardware_capture" / "rf" / "redstone"
EXAMPLES_DIR = REDSTONE_DIR / "examples"
OUTPUT_DIR = EXAMPLES_DIR / "generated"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Output dir:", OUTPUT_DIR)

from hardware_capture.rf.redstone import Sequence

## 2. Select Files And Export Options

`include_raw_bytes=True` preserves the full TPS payload in base64 for debugging. Turn it off for smaller JSON files.

In [ ]:
INPUT_PATH = (
    REDSTONE_DIR
    / "references"
    / "examples"
    / "tnmr-sequences"
    / "common"
    / "1pulse - f1.tps"
)
OUTPUT_PATH = OUTPUT_DIR / f"{INPUT_PATH.name}.json"
INCLUDE_STRINGS = False
INCLUDE_RAW_BYTES = True
INCLUDE_SEQUENCE_ROWS_RAW = True

assert INPUT_PATH.exists(), INPUT_PATH
INPUT_PATH, OUTPUT_PATH

## 3. Load And Summarize

In [ ]:
sequence = Sequence.load(INPUT_PATH)
sequence.summary().to_dict()

## 4. Write JSON

In [ ]:
payload = sequence.to_json(
    include_strings=INCLUDE_STRINGS,
    include_raw_bytes=INCLUDE_RAW_BYTES,
    include_sequence_rows_raw=INCLUDE_SEQUENCE_ROWS_RAW,
)
OUTPUT_PATH.write_text(payload + "\n", encoding="utf-8")
print("Wrote JSON:", OUTPUT_PATH)
print("Characters:", len(payload))

## 5. Quick Preview

In [ ]:
print(payload[:2000])